In [3]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")
graph_repo = AlarmGraphRepository(os.getenv("HISTORY_DB_PATH"))

In [4]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationHistory

SimpleTimeCorrelationHistory.train(lazy_frame, graph_repo)

Processando nós: 100%|██████████| 912/912 [07:22<00:00,  2.06nó/s] 


In [5]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

Calculando WCC: 100%|██████████| 912/912 [03:57<00:00,  3.85nó/s] 


alert_id,node_id,alert_type,start_time,end_time,incident
str,str,str,date,date,i32
"""f731a04b-5e80-4fc2-94b9-49eefd…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-29,null,3881
"""9b6cb823-0895-41d5-abd0-8ec2ee…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-07,null,875
"""0ea25bc5-ae0f-4b91-a69a-970925…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-09,null,null
"""83d1c7e9-1a90-4d9b-aa87-ba0fc0…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-08,null,981
"""d1375fa2-2d33-4753-891b-79e71e…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-28,null,3715
…,…,…,…,…,…
"""e3b5cdf9-49fd-4161-8aaf-5f9a6f…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-10,null,1282
"""e6b77010-1fa9-458e-8e73-9ab97d…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-23,null,3054
"""93043f74-5865-4b46-89da-f1b0a0…","""50e91219-3c5e-420d-9c21-0c80cd…","""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-22,null,2813


In [6]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_history")
results_repo.save(summary)
results_repo.load()

✅ Nova versão salva com sucesso em: data/results/simple_time_corr_history_20260520_170147.csv
📖 Carregando a versão mais recente encontrada: simple_time_corr_history_20260520_170147.csv


Node ID,Total de Alarmes,Total de Correlações,Total de Incidentes,Média de Alarmes por Incidente,Densidade
str,i64,i64,i64,f64,f64
"""a5126d4f-0bd1-4023-ae9d-d2c1e9…",232712,8584164,4090,56.8978,0.000317
"""7510d37a-4c5f-40da-b987-3977c2…",32148,119403,4073,7.892954,0.000231
"""8bf0d4d2-1cd3-4341-9b79-794833…",31766,116570,4065,7.814514,0.000231
"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…",32023,119030,4054,7.899112,0.000232
"""75577cb5-6565-42da-a042-d9603e…",27157,83723,4041,6.720366,0.000227
…,…,…,…,…,…
"""2ccfd5c5-2244-4e8e-980e-7ad5e2…",242,0,1,242.0,0.0
"""70698859-ea2b-41be-a0d8-d29f4c…",11,0,1,11.0,0.0
"""44837bb2-30ba-4375-9303-497045…",755,0,1,755.0,0.0
